# Plotting Parrots

## Initialising codebase

In [2]:
import sys
import os
from datetime import datetime as dt
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Import data

In [3]:

beaufort_scale = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/beaufort_scale.csv')
birds = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/birds.csv')
sea_states = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/sea_states.csv')
ships = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/ships.csv')

C:\Users\jason\AppData\Local\Temp\ipykernel_3180\3598388994.py:2: DtypeWarning: Columns (0: sex) have mixed types. Specify dtype option on import or set low_memory=False.
  birds = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/birds.csv')


### Check if data imported correctly

In [4]:
beaufort_scale.head()

,wind_speed_class,wind_description,wind_speed_knots_min,wind_speed_knots_max
0,0,calm,0,1.0
1,1,light air,1,3.0
2,2,light breeze,4,6.0
3,3,gentle breeze,7,10.0
4,4,moderate breeze,11,16.0


In [5]:
birds.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,sitting_on_ship,in_hand,n_flying_past,flying_past,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,False,False,0.0,False,6.0,True,0.0,False,NaN,False
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,False,False,0.0,False,8.0,True,0.0,False,NaN,False
3,4,1083001,Fairy prion,Pachyptila turtur,PACTUR,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,False,False,0.0,False,4.0,True,0.0,False,NaN,False


In [15]:
birds_new = birds.copy()

birds_new["Genus"] = birds_new["species_scientific_name"].str.split().str[0]
birds_new.head()
birds_new["Genus"].value_counts()
birds_new["Genus"].nunique()

47

### Data cleaning: genus names
According to IOC World Bird List v15.2 to standardise the data

In [ ]:
# Misspelt genus
GENUS_TYPOS = {
    "Thallassarche": "Thalassarche",
    "Phaeton":       "Phaethon",
    "Magadyptes":    "Megadyptes",
}

# Genus renamed in the IOC World Bird List
IOC_GENUS_RENAMES = {
    "Catharacta":   "Stercorarius",
    "Stictocarbo":  "Phalacrocorax",
    "Procelsterna": "Anous",
    "Lugensa":      "Aphrodroma",
}

# Changed genera by IOC
IOC_SPECIES_MOVES = {
    "Diomedea impavida":     "Thalassarche",
    "Diomedea melanophrys":  "Thalassarche",
    "Puffinus griseus":      "Ardenna",
    "Puffinus tenuirostris": "Ardenna",
    "Puffinus carneipes":    "Ardenna",
    "Puffinus pacificus":    "Ardenna",
    "Puffinus bulleri":      "Ardenna",
    "Larus novaehollandiae": "Chroicocephalus",
    "Larus bulleri":         "Chroicocephalus",
    "Sterna bergii":         "Thalasseus",
    "Sterna albifrons":      "Sternula",
    "Sterna nereis":         "Sternula",
}

def ioc_genera(name):
    genera, genus = set(), None
    for part in name.split("/"):
        words = part.split()
        if not words:
            continue
        if words[0][0].isupper():   # part names its own genus: "Genus species"
            genus, epithet = words[0], (words[1] if len(words) > 1 else "")
        else:                       # "Genus a / b": b shares the previous genus
            epithet = words[0]
        fixed = GENUS_TYPOS.get(genus, IOC_GENUS_RENAMES.get(genus, genus))
        genera.add(IOC_SPECIES_MOVES.get(f"{genus} {epithet}", fixed))
    return sorted(genera)   # unique genera, as a list

# Drop unidentified bird species
birds_new = birds_new.dropna(subset=["species_scientific_name"]).copy()

# Drop names that span two genera
genera = birds_new["species_scientific_name"].apply(ioc_genera)
spans = genera.apply(len) > 1
print("Dropped (span two genera):\n", birds_new.loc[spans, "species_scientific_name"].value_counts(), "\n")
birds_new = birds_new[~spans]

# Assign IOC genus (typos and renames applied)
birds_new["Genus_original"] = birds_new["Genus"]
birds_new["Genus"] = genera[~spans].str[0]

changed = birds_new[birds_new["Genus"] != birds_new["Genus_original"]]
print(changed.groupby(["Genus_original", "Genus"]).size().rename("rows"))
print("\nRows remaining:", len(birds_new), "| Genera:", birds_new["Genus"].nunique())

In [23]:
# Count each unique genus
genus_by_count = birds_new.groupby(["record_id", "Genus"], as_index=False)["count"].sum()

birds_new.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,n_flying_past,flying_past,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding,Genus,Genus_original
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,0.0,False,6.0,True,0.0,False,NaN,False,Diomedea,Diomedea
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,0.0,False,2.0,True,0.0,False,NaN,False,Thalassarche,Diomedea
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,0.0,False,8.0,True,0.0,False,NaN,False,Daption,Daption
3,4,1083001,Fairy prion,Pachyptila turtur,PACTUR,NaN,NaN,NaN,NaN,2.0,...,0.0,False,2.0,True,0.0,False,NaN,False,Pachyptila,Pachyptila
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,0.0,False,4.0,True,0.0,False,NaN,False,Ardenna,Puffinus


In [ ]:
sea_states.head()

,sea_state_class,sea_state_description,wave_meters_min,wave_meters_max
0,0,"calm, glassy",0.0,0.0
1,1,"calm, rippled",0.0,0.1
2,2,smooth,0.1,0.5
3,3,slight,0.5,1.2
4,4,moderate,1.3,2.5


In [ ]:
ships.head()

,record_id,date,time,latitude,longitude,hemisphere,activity,speed,direction,cloud_cover,...,wind_speed_class,wind_direction,air_temperature,pressure,sea_state_class,sea_surface_temperature,depth,observer,census_method,season
0,1083001,1975-10-15,14:00:00,-45.917,165.400,E,"steaming, sailing",15.0,NaN,overcast,...,5.0,NaN,NaN,NaN,5.0,NaN,NaN,D. Jeffcock,full,spring
1,1084001,1975-11-03,13:10:00,-35.533,125.000,E,"steaming, sailing",14.0,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
2,1084002,1975-11-04,14:20:00,-37.667,132.250,E,"steaming, sailing",14.5,NaN,overcast,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
3,1084003,1975-11-08,16:15:00,-40.000,162.000,E,"steaming, sailing",14.6,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
4,1086001,1975-11-16,12:30:00,-36.167,174.917,E,"steaming, sailing",15.0,NaN,partially cloudy,...,0.0,NaN,NaN,NaN,1.0,NaN,NaN,D. Jeffcock,full,spring
